In [20]:
from AgentBasedModel.params_calibration.calibrationv2.utils_calibration_v2 import *
import pandas as pd
import random

In [21]:
file_10s = "ethusdt_10s_300days.csv"
prices_10s = get_prices(file_10s)

SLICE = 1000
num = ['mean_on_ret2', "std_on_ret2", "q1_on_ret2", "q5_on_ret2", "q95_on_ret2", "q99_on_ret2", "kurtosis_on_ret2", "skewness_on_ret"]
arrs = ["autocorrelation_on_ret", "autocorrelation_on_abs_ret"]

In [22]:
# посчитать среднее по метрикам из реального распределения + матрицу ковариаций, чтобы получился хорошая функция loss (нормированная адекватно) из статьи WINKER
df = pd.DataFrame()
step = 0
for i in range(0, prices_10s.shape[0] - SLICE + 1, SLICE):
    step += 1
    if step % 1000 == 0:
        print(step)
    cur_start = i
    cur_slice = prices_10s[cur_start:cur_start+SLICE]
    cur_params = pipeline(cur_slice, 0)
    ind = len(df)
    df.loc[ind, "start"] = int(cur_start)
    for key, val in cur_params.items():
        if key in arrs:
            idd = 0
            for v in val:
                df.loc[ind, f"{key}_lag_{idd}"] = v
                idd += 1
        else:
            df.loc[ind, key] = val

df

1000
2000
3000


,start,mean_on_ret2,std_on_ret2,q1_on_ret2,q5_on_ret2,q95_on_ret2,q99_on_ret2,kurtosis_on_ret2,skewness_on_ret,autocorrelation_on_ret_lag_0,...,autocorrelation_on_abs_ret_lag_11,autocorrelation_on_abs_ret_lag_12,autocorrelation_on_abs_ret_lag_13,autocorrelation_on_abs_ret_lag_14,autocorrelation_on_abs_ret_lag_15,autocorrelation_on_abs_ret_lag_16,autocorrelation_on_abs_ret_lag_17,autocorrelation_on_abs_ret_lag_18,autocorrelation_on_abs_ret_lag_19,autocorrelation_on_abs_ret_lag_20
0,0.0,0.001426,0.001843,-0.004361,-0.003456,0.001795,0.003647,0.336559,-0.193673,1.0,...,0.119309,0.131280,0.140567,0.111542,0.004795,0.018528,0.096622,0.144538,0.086544,0.061965
1,1000.0,0.001496,0.001938,-0.002484,-0.001252,0.004174,0.006609,2.041160,0.870470,1.0,...,-0.012385,0.018552,0.066056,0.039604,0.037784,0.085109,0.018699,0.050597,-0.019655,-0.002241
2,2000.0,0.001130,0.001609,-0.001630,-0.001346,0.002068,0.005810,8.071705,4.398720,1.0,...,0.014964,0.017612,0.030693,-0.012347,0.009703,-0.008834,-0.004729,0.016153,-0.016108,-0.002906
3,3000.0,0.001062,0.001362,-0.003035,-0.001840,0.002345,0.002978,0.245135,0.055022,1.0,...,0.043935,0.042871,0.057569,0.055318,0.039006,0.031446,0.041744,0.030544,0.002303,0.025123
4,4000.0,0.001222,0.001491,-0.003281,-0.002974,0.002096,0.002440,-0.486196,-0.777378,1.0,...,0.121516,0.074327,0.115339,0.069197,0.035045,0.050702,0.123913,0.045469,0.075038,0.073642
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3157,3157000.0,0.001067,0.001349,-0.001810,-0.001620,0.002677,0.003310,-0.163621,0.232862,1.0,...,0.091022,0.059543,0.049965,0.017811,-0.004811,0.028682,0.004679,0.034973,-0.005777,-0.005268
3158,3158000.0,0.001584,0.002284,-0.007598,-0.004631,0.002367,0.002893,3.402873,0.899748,1.0,...,0.128873,0.143992,0.126343,0.044698,0.106283,0.091494,0.109281,0.126938,0.114738,0.090019
3159,3159000.0,0.001631,0.002017,-0.004660,-0.003722,0.002990,0.003802,-0.127846,-0.640254,1.0,...,0.066239,0.063965,0.033432,0.055764,0.101049,0.023624,-0.008351,0.060027,0.010160,0.010773
3160,3160000.0,0.001294,0.001871,-0.001596,-0.001244,0.003804,0.006837,4.428781,0.030910,1.0,...,0.165096,0.169803,0.124955,0.120895,0.131950,0.176169,0.056123,0.113525,0.077405,0.151094


In [23]:
# более аккуратная работа с автокорреляцией: выделено 3 основных: 1, 5, 10
ret_ac_cols = [c for c in df.columns if c.startswith("autocorrelation_on_ret_lag_")]
abs_ac_cols = [c for c in df.columns if c.startswith("autocorrelation_on_abs_ret_lag_")]

base_cols = ['mean_on_ret2', "std_on_ret2", "q1_on_ret2", "q5_on_ret2", "q95_on_ret2", "q99_on_ret2", "kurtosis_on_ret2", "skewness_on_ret", ]

df_summary = df[num].copy()
df_summary["autocorr_on_ret_1"] = df[ret_ac_cols[1]]
df_summary["autocorr_on_ret_5"] = df[ret_ac_cols[5]]
df_summary["autocorr_on_ret_10"] = df[ret_ac_cols[10]]
df_summary["autocorr_on_abs_1"] = df[abs_ac_cols[1]]
df_summary["autocorr_on_abs_5"] = df[abs_ac_cols[5]]
df_summary["autocorr_on_abs_10"] = df[abs_ac_cols[10]]

sigma_df = df_summary.cov()

sigma_df

,mean_on_ret2,std_on_ret2,q1_on_ret2,q5_on_ret2,q95_on_ret2,q99_on_ret2,kurtosis_on_ret2,skewness_on_ret,autocorr_on_ret_1,autocorr_on_ret_5,autocorr_on_ret_10,autocorr_on_abs_1,autocorr_on_abs_5,autocorr_on_abs_10
mean_on_ret2,5.853279e-07,8.046958e-07,-0.000002,-0.000001,0.000001,0.000002,0.000178,0.000015,0.000008,0.000017,0.000017,0.000017,0.000017,0.000015
std_on_ret2,8.046958e-07,1.164215e-06,-0.000003,-0.000002,0.000002,0.000003,0.000632,0.000003,0.000008,0.000020,0.000019,0.000032,0.000030,0.000026
q1_on_ret2,-1.784101e-06,-2.647201e-06,0.000008,0.000004,-0.000003,-0.000005,-0.001844,0.000774,-0.000021,-0.000048,-0.000047,-0.000087,-0.000079,-0.000066
q5_on_ret2,-1.156335e-06,-1.603663e-06,0.000004,0.000003,-0.000002,-0.000003,-0.000257,0.000243,-0.000021,-0.000040,-0.000040,-0.000037,-0.000038,-0.000033
q95_on_ret2,1.225201e-06,1.730944e-06,-0.000003,-0.000002,0.000003,0.000005,0.000368,0.000304,0.000009,0.000026,0.000024,0.000037,0.000035,0.000032
q99_on_ret2,1.837526e-06,2.760662e-06,-0.000005,-0.000003,0.000005,0.000008,0.002310,0.000725,0.000005,0.000027,0.000022,0.000087,0.000077,0.000067
kurtosis_on_ret2,1.777042e-04,6.323392e-04,-0.001844,-0.000257,0.000368,0.002310,4.919474,-0.103652,-0.028665,-0.028172,-0.034756,0.093032,0.072543,0.056907
skewness_on_ret,1.497850e-05,2.873989e-06,0.000774,0.000243,0.000304,0.000725,-0.103652,1.964282,-0.000418,-0.002659,-0.004863,-0.009879,-0.003516,-0.001019
autocorr_on_ret_1,8.283853e-06,8.270175e-06,-0.000021,-0.000021,0.000009,0.000005,-0.028665,-0.000418,0.003311,0.002034,0.002107,-0.001511,-0.000831,-0.000620
autocorr_on_ret_5,1.733065e-05,1.954855e-05,-0.000048,-0.000040,0.000026,0.000027,-0.028172,-0.002659,0.002034,0.003399,0.002581,-0.001451,-0.000497,-0.000469


In [14]:
sigma_inv = np.linalg.inv(sigma_df.to_numpy())
sigma_inv_df = pd.DataFrame(sigma_inv, index=sigma_df.columns, columns=sigma_df.columns)

sigma_inv_df

,mean_on_ret2,std_on_ret2,q1_on_ret2,q5_on_ret2,q95_on_ret2,q99_on_ret2,kurtosis_on_ret2,skewness_on_ret,autocorr_on_ret_1,autocorr_on_ret_5,autocorr_on_ret_10,autocorr_on_abs_1,autocorr_on_abs_5,autocorr_on_abs_10
mean_on_ret2,1.124994e+08,-1.225042e+08,-1.051585e+07,2.314637e+06,-6.070339e+05,1.016808e+07,2739.641856,-434.050340,28362.098024,-31461.508637,-34645.832169,7435.927132,-13780.934566,16217.545218
std_on_ret2,-1.225042e+08,2.113546e+08,2.188186e+07,6.972520e+06,-7.923861e+06,-2.249639e+07,-2868.832861,534.774216,-38556.580948,23187.315325,16526.037485,-11865.670799,15290.809780,-10226.949045
q1_on_ret2,-1.051585e+07,2.188186e+07,3.383111e+06,-2.843063e+05,-4.713146e+05,-2.836902e+06,166.639184,-120.453166,-3045.502888,1246.039063,329.934117,-585.446848,1515.326050,-101.278266
q5_on_ret2,2.314637e+06,6.972520e+06,-2.843063e+05,3.890821e+06,-1.553933e+06,-6.034192e+05,-521.176484,52.998009,-957.843603,2774.981194,1481.059055,977.006917,1226.213593,1954.028319
q95_on_ret2,-6.070339e+05,-7.923861e+06,-4.713146e+05,-1.553933e+06,3.519951e+06,-2.267854e+05,657.338023,-19.296636,3420.274984,-46.856305,2896.239825,803.042740,179.418021,-810.874069
q99_on_ret2,1.016808e+07,-2.249639e+07,-2.836902e+06,-6.034192e+05,-2.267854e+05,3.660666e+06,-255.964506,-181.167193,2553.771815,-620.711496,-234.511144,534.139660,-1271.646606,-147.147970
kurtosis_on_ret2,2.739642e+03,-2.868833e+03,1.666392e+02,-5.211765e+02,6.573380e+02,-2.559645e+02,0.636701,-0.000041,1.340865,-1.133115,0.737306,-1.139309,-1.947162,-0.047041
skewness_on_ret,-4.340503e+02,5.347742e+02,-1.204532e+02,5.299801e+01,-1.929664e+01,-1.811672e+02,-0.000041,0.630562,-0.706669,-0.032207,1.240192,1.263809,-0.325767,-0.886598
autocorr_on_ret_1,2.836210e+04,-3.855658e+04,-3.045503e+03,-9.578436e+02,3.420275e+03,2.553772e+03,1.340865,-0.706669,553.663631,-181.420566,-197.956023,23.418762,19.220718,-0.694880
autocorr_on_ret_5,-3.146151e+04,2.318732e+04,1.246039e+03,2.774981e+03,-4.685631e+01,-6.207115e+02,-1.133115,-0.032207,-181.420566,794.237431,-378.372877,82.563315,-62.229978,31.949668


In [19]:
sigma_inv_df.to_csv("weights(inv_covariance_matrix_from_real_data).csv", index=False)

In [17]:
df2 = (
    df_summary
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={"index": "param"})
)

df2.to_csv("mean_std_ethusdt_validation_data2.csv", index=False)

In [18]:
df2

,param,mean,std
0,mean_on_ret2,0.001396,0.000765
1,std_on_ret2,0.001836,0.001079
2,q1_on_ret2,-0.003937,0.002800
3,q5_on_ret2,-0.002725,0.001691
4,q95_on_ret2,0.002753,0.001799
5,q99_on_ret2,0.004000,0.002894
6,kurtosis_on_ret2,1.054911,2.217989
7,skewness_on_ret,-0.011446,1.401528
8,autocorr_on_ret_1,0.321266,0.057539
9,autocorr_on_ret_5,0.294196,0.058298


In [26]:
sigma_diag = np.diag(np.diag(sigma_df.to_numpy()))
sigma_diag_df = pd.DataFrame(np.linalg.inv(sigma_diag), index=sigma_df.columns, columns=sigma_df.columns)
sigma_diag_df.to_csv("weights(inv only diag).csv", index=False)

In [27]:
sigma_diag_df

,mean_on_ret2,std_on_ret2,q1_on_ret2,q5_on_ret2,q95_on_ret2,q99_on_ret2,kurtosis_on_ret2,skewness_on_ret,autocorr_on_ret_1,autocorr_on_ret_5,autocorr_on_ret_10,autocorr_on_abs_1,autocorr_on_abs_5,autocorr_on_abs_10
mean_on_ret2,1.708444e+06,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
std_on_ret2,0.000000e+00,858947.622751,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
q1_on_ret2,0.000000e+00,0.000000,127508.21941,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
q5_on_ret2,0.000000e+00,0.000000,0.00000,349772.145737,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
q95_on_ret2,0.000000e+00,0.000000,0.00000,0.000000,308946.93887,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
q99_on_ret2,0.000000e+00,0.000000,0.00000,0.000000,0.00000,119426.804263,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
kurtosis_on_ret2,0.000000e+00,0.000000,0.00000,0.000000,0.00000,0.000000,0.203274,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
skewness_on_ret,0.000000e+00,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.509092,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000
autocorr_on_ret_1,0.000000e+00,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,302.04907,0.000000,0.000000,0.000000,0.000000,0.00000
autocorr_on_ret_5,0.000000e+00,0.000000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,294.231191,0.000000,0.000000,0.000000,0.00000
